# AI Builders Club - NetworkEvolution | Session 1
## Script 04 : Chat with your LIVE network  (script 03 + MEMORY)

- **Script 03** = a question loop, but every question is independent. *"and on sw2?"* fails - the AI forgot the last question.
- **Script 04** = a real **chat**. The AI **remembers** the conversation: ask, read, follow up
  (*"and on sw2?"*, *"show it as a list"*, *"why is that risky?"*).

What changes:
- same start as 03: sample questions first, then your question
- **first** question -> SSH to sw1 + sw2 once, send **configs + question**
- **follow-ups** -> send **only the question** + the conversation id
- **`refresh`** -> SSH again + start a fresh conversation

| Section | What you build | New? |
|---------|----------------|------|
| 1-4 | Imports, switch dicts, `get_running_config()`, `collect_configs()` | same as 03 |
| 5 | `SYSTEM_INSTRUCTION` | **+1 rule** |
| 6 | Sample questions | **+ "and on sw2?"** and `refresh` |
| 7 | `ask_ai(text, previous_id)` | **+ `previous_interaction_id`** |
| 8 | The chat loop | **+ memory + refresh** |

New lines in the code are marked `# NEW (04)`.

### How to use this notebook
Every **code cell** below is one section of the script `04_chat_with_network.py`.

1. Create an empty file `04_chat_with_network.py` in your session folder.
2. Copy the code cells **one by one, in order**, and paste each one at the **bottom** of that file.
3. After each paste, run it: `python 04_chat_with_network.py` - and check the result described above the cell.

> Markdown cells (like this one) are the explanations - do **not** paste them into the `.py` file.

### Big picture: how does the AI "remember"?
An LLM itself has **no memory** - every call starts from zero. Two ways to build a chat:

- **A)** **You** resend the whole history every time (classic chat APIs)
- **B)** The **server** keeps the history -> **Interactions API** (we use this)

```
Turn 1 : create(input = configs + question 1)                    -> id "A"
Turn 2 : create(input = question 2, previous_interaction_id="A") -> id "B"
Turn 3 : create(input = question 3, previous_interaction_id="B") -> id "C"
```
Each interaction gets an `.id`. Pass the **last** id as `previous_interaction_id` -> Google adds everything before it
(configs, questions, answers) for you.

> **Watch the token counter:** input tokens still **grow** every turn. The model re-reads the history each time -
> server-side memory saves you **uploading** it, not **paying** for it.

### Program flow (differences to 03 marked with `*`)
```
start -> show SAMPLE QUESTIONS
  |
  v
+--> "You:"  <------------------------------------------------+
|      +-- "exit"     -> stop                                 |
|      +-- "help"     -> show samples again ------------------+
|    * +-- "refresh"  -> SSH again, previous_id = None -------+
|      +-- 1..8       -> pick that sample question            |
|      v                                                      |
|    configs collected already?  NO -> SSH to sw1 + sw2 once  |
|      v                                                      |
|  * previous_id is None?  (no conversation yet)              |
|      YES -> input = configs + question     (~2,500 tokens)  |
|      NO  -> input = question ONLY + previous_interaction_id |
|      v                                                      |
|    Gemini -> print answer                                   |
|  * previous_id = interaction.id  (remember this turn) ------+
```

### Key words
| Word | Meaning |
|------|---------|
| **Multi-turn** | A conversation with more than one question |
| **`interaction.id`** | The name of this turn on Google's side |
| **`previous_interaction_id`** | "Continue from this turn" -> memory. `None` = start a **new** conversation |
| **`store=True`** (default) | Google stores the interaction so it can be continued (for a limited time). `store=False` -> nothing stored, no chat |
| **Stale data** | The AI only knows the configs from the moment we collected them. Changed a switch? -> `refresh` |

---
## Setup (one time - in the terminal, not in the `.py` file)

Skip this if you already did it for the earlier scripts.

```bash
pip install -r requirements.txt
cp .env.example .env
```
Your `.env` must contain your API key **and** the switch details:
```
GEMINI_API_KEY=paste-your-key-here
DEVICE_USERNAME=admin
DEVICE_PASSWORD=admin
SW1_HOST=172.20.20.11
SW2_HOST=172.20.20.12
# optional, default 22:
# SW1_PORT=22
# SW2_PORT=22
```
> `.env` holds secrets -> **never** share it, **never** commit it to git.

---
## Section 1 : Imports + load the `.env` file  (same as script 03)

In [ ]:
# ---------------------------------------------------------------
# SECTION 1 : Imports + load the .env file  (same as 03)
# ---------------------------------------------------------------
import os

from dotenv import load_dotenv
from google import genai
from netmiko import ConnectHandler

load_dotenv()

---
## Section 2 : One dictionary per switch  (same as script 03)

In [ ]:
# ---------------------------------------------------------------
# SECTION 2 : One dictionary per switch  (same as 03)
# ---------------------------------------------------------------

USERNAME = os.getenv("DEVICE_USERNAME")
PASSWORD = os.getenv("DEVICE_PASSWORD")

SWITCHES = {
    "sw1": {
        "device_type": "arista_eos",
        "host": os.getenv("SW1_HOST"),
        "port": int(os.getenv("SW1_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
    "sw2": {
        "device_type": "arista_eos",
        "host": os.getenv("SW2_HOST"),
        "port": int(os.getenv("SW2_PORT", 22)),
        "username": USERNAME,
        "password": PASSWORD,
    },
}

---
## Section 3 : SSH to one switch, return its config  (same as script 03)

In [ ]:
# ---------------------------------------------------------------
# SECTION 3 : SSH to ONE switch and return its running config  (same as 03)
# ---------------------------------------------------------------

def get_running_config(device):
    with ConnectHandler(**device) as conn:
        conn.enable()
        return conn.send_command("show running-config")

---
## Section 4 : Collect BOTH configs into ONE text  (same as script 03)

Each config gets a label `=== DEVICE: sw1 ===` so the AI knows which switch a line belongs to.

In [ ]:
# ---------------------------------------------------------------
# SECTION 4 : Collect BOTH configs into ONE text  (same as 03)
# ---------------------------------------------------------------

def collect_configs():
    parts = []
    for name, device in SWITCHES.items():
        print(f"Getting running config from {name} ({device['host']}) ...")
        try:
            config = get_running_config(device)
        except Exception as error:
            print(f"  {name}: FAILED -> {type(error).__name__}: {error}")
            continue

        print(f"  {name}: {len(config.splitlines())} lines")
        parts.append(f"=== DEVICE: {name} ===\n{config}")

    return "\n\n".join(parts)

---
## Section 5 : Gemini client + model + system instruction  - ONE new rule

Same as script 03, plus one rule at the end:

> *"This is a conversation: follow-up questions refer to the same configs and to your previous answers."*

Without it the model may treat *"and on sw2?"* as a brand-new, unclear question.

In [ ]:
# ---------------------------------------------------------------
# SECTION 5 : Gemini client + model + system instruction  (03 + 1 rule)
# ---------------------------------------------------------------
client = genai.Client()
MODEL = "gemini-3.8-flash"

SYSTEM_INSTRUCTION = """
You are a senior network engineer. You receive the LIVE running
configuration of Arista EOS switches, collected by a script a few
seconds ago. Each config starts with a line like "=== DEVICE: sw1 ===".

Rules:
- Answer ONLY from the configs you received. Do not guess.
- If the answer is not in the configs, say: "I don't see that in the config."
- Always say which device (sw1 / sw2) each fact comes from.
- Plain text for a terminal: short lines or "- " bullets.
  No markdown: no **bold**, no # headings, no tables.
- Keep it short (max 12 lines) unless the user asks for more detail.
- This is a conversation: follow-up questions refer to the same configs
  and to your previous answers.
"""                                                             # NEW (04)

---
## Section 6 : Sample questions  - + "and on sw2?" + refresh

- Question **8** = *"and on sw2?"* -> the **memory test**. Ask **3** first, then **8**.
- `show_samples()` gets one more line explaining the `refresh` command.

In [ ]:
# ---------------------------------------------------------------
# SECTION 6 : Sample questions + show_samples()  (03 + memory test + refresh)
# ---------------------------------------------------------------
SAMPLE_QUESTIONS = [
    "Which VLANs are configured on each switch? Show them with names.",
    "Compare sw1 and sw2. What is different?",
    "Which interfaces on sw1 have an IP address?",
    "Which users can log in and with which privilege level?",
    "Is any interface shutdown?",
    "Explain the sw1 config to a junior engineer in 5 lines.",
    "Is BGP configured anywhere?",
    "and on sw2?",                                              # NEW (04) needs memory!
]


def show_samples():
    print("\n" + "#" * 64)
    print("#  SAMPLE QUESTIONS  - type a NUMBER or your own question")
    print("#" * 64)
    for number, question in enumerate(SAMPLE_QUESTIONS, start=1):
        print(f"   {number}. {question}")
    print("-" * 64)
    print("   help = show these questions again      exit = quit")
    print("   refresh = get the live configs again + start a new conversation")   # NEW (04)
    print("#" * 64)

---
## Section 7 : ONE turn of the conversation  - + memory

Same `create()` call as script 03, plus **one new argument**:

| `previous_interaction_id` | Meaning |
|---------------------------|---------|
| `None` | start a **new** conversation |
| `"v1_..."` (an earlier `interaction.id`) | **continue** that conversation |

- `text` is what we send **this** turn - the loop decides: first turn -> configs + question, follow-up -> question only.
- We return the whole `interaction` because we need its `.id` for the next turn.

In [ ]:
# ---------------------------------------------------------------
# SECTION 7 : ONE turn of the conversation  (03 + memory)
# ---------------------------------------------------------------
def ask_ai(text, previous_id):                                  # NEW (04)
    return client.interactions.create(
        model=MODEL,
        input=text,
        system_instruction=SYSTEM_INSTRUCTION,
        generation_config={"thinking_level": "low"},
        previous_interaction_id=previous_id,                    # NEW (04)
    )

---
## Section 8 : The chat loop  - + memory + refresh (the program starts here)

New compared to script 03:
- `previous_id = None` -> no conversation yet
- **First turn:** `text = configs + question`
- **Follow-ups:** `text = question` only - Google already has the configs in the history
- After every answer: `previous_id = interaction.id` -> remember this turn
- **`refresh`:** SSH again **and** `previous_id = None` - a new conversation, because the old one is based on **old** configs
- `collect_configs() or None` -> if both switches fail (`""`), store `None` so the next question tries again

**Run the full script now** and try this conversation:
| Type | What to observe |
|------|-----------------|
| `3` | interfaces with an IP on sw1 |
| `8` | *"and on sw2?"* -> **works now!** (memory) |
| `which of those are links between sw1 and sw2?` | uses **both** previous answers |
| *(on sw1:* `vlan 999` / `name TEST`*)* then `refresh` | fresh configs + new conversation |
| `Which VLANs are on sw1 but not on sw2?` | sees VLAN 999 (then remove it: `no vlan 999`) |
| `exit` | quit |

Watch the **input tokens** - they keep growing every turn, even though we only send the question.

In [ ]:
# ---------------------------------------------------------------
# SECTION 8 : The chat loop  (03 + memory + refresh)
# ---------------------------------------------------------------
show_samples()
configs_text = None
previous_id = None                                              # NEW (04)

while True:
    question = input("\nYou: ").strip()

    # --- 8a : commands -------------------------------------------
    if not question:
        continue
    if question.lower() in ("exit", "quit"):
        print("Bye!")
        break
    if question.lower() == "help":
        show_samples()
        continue
    if question.lower() == "refresh":                           # NEW (04)
        print()
        configs_text = collect_configs() or None                # "" (all failed) -> None
        previous_id = None                                      # new configs -> new conversation
        print("(fresh configs collected - new conversation started)")
        continue
    if question.isdigit() and 1 <= int(question) <= len(SAMPLE_QUESTIONS):
        question = SAMPLE_QUESTIONS[int(question) - 1]
        print(f"You: {question}")

    # --- 8b : first question -> collect the configs ONCE ---------
    if configs_text is None:
        print()
        configs_text = collect_configs()
        if not configs_text:
            print("No config collected - check .env and the lab, then ask again.")
            configs_text = None
            continue

    # --- 8c : first turn = configs + question, follow-up = question only
    if previous_id is None:                                     # NEW (04) new conversation
        text = f"{configs_text}\n\nQUESTION: {question}"
    else:                                                       # NEW (04) follow-up
        text = question

    # --- 8d : ask Gemini, REMEMBER the id, print answer + tokens --
    print("\nAsking Gemini ...\n")
    try:
        interaction = ask_ai(text, previous_id)                 # NEW (04)
    except Exception as error:
        print(f"AI call failed -> {type(error).__name__} "
              f"(HTTP {getattr(error, 'status_code', '-')}): {str(error)[:300]}")
        continue

    previous_id = interaction.id                                # NEW (04) remember this turn
    print(f"AI: {interaction.output_text}")

    usage = interaction.usage
    print(f"\n[tokens -> input: {usage.total_input_tokens}, "
          f"output: {usage.total_output_tokens}, "
          f"thinking: {usage.total_thought_tokens}]")

---
## Recap - what you can now explain

- An LLM has **no memory** - a chat is built by sending history, or by letting the **server** keep it.
- With the Interactions API: `previous_interaction_id = <last interaction.id>` continues the conversation.
- Follow-ups send only the **question** - but **input tokens still grow**: the model re-reads the history every turn.
- Data in the conversation is a **snapshot** -> `refresh` collects new configs and starts a new conversation.

### Where this goes next (full course)
Today the AI only **reads**. In the **AI Automation course for Network Engineers** you go further:
tool calling (the AI picks which command to run), secret sanitization, config generation with Jinja2 + validation,
RAG over all your backups, AI agents with Google ADK, and your own **MCP server** for VLAN management.